# 📊 Model Evaluation

## Objective

Evaluate the Linear Regression baseline using a group-aware train/test split.

This notebook:

- uses the refined dataset
- prevents feature-group overlap between train and test
- trains Linear Regression inside a preprocessing pipeline
- evaluates using MAE, MSE, RMSE, and R²

No hyperparameter tuning is performed in this notebook.

In [1]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv(
    "../data/processed/house_prices_refined.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (13297, 5)


In [3]:
feature_columns = [
    "bhk",
    "propertytype",
    "location",
    "sqft"
]

target_column = "totalprice"

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Features:", feature_columns)
print("Target:", target_column)

Features: ['bhk', 'propertytype', 'location', 'sqft']
Target: totalprice


In [4]:
groups = (
    df[feature_columns]
    .astype(str)
    .agg("||".join, axis=1)
)

print("Total rows:", len(df))
print("Unique feature groups:", groups.nunique())

Total rows: 13297
Unique feature groups: 9359


In [5]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (10556, 4)
X_test shape: (2741, 4)
y_train shape: (10556,)
y_test shape: (2741,)


In [6]:
train_groups = set(groups_train)
test_groups = set(groups_test)

overlap = train_groups.intersection(test_groups)

print("Unique feature groups in train:", len(train_groups))
print("Unique feature groups in test:", len(test_groups))
print("Overlapping feature groups:", len(overlap))

assert len(overlap) == 0, "Feature-group leakage detected!"

print("No feature-group overlap detected.")

Unique feature groups in train: 7487
Unique feature groups in test: 1872
Overlapping feature groups: 0
No feature-group overlap detected.


In [7]:
categorical_features = [
    "propertytype",
    "location"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [8]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [9]:
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [10]:
predictions = model.predict(X_test)

print("Predictions generated:", len(predictions))

Predictions generated: 2741


In [11]:
mae = mean_absolute_error(
    y_test,
    predictions
)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = mse ** 0.5

r2 = r2_score(
    y_test,
    predictions
)

print(f"MAE : {mae:,.2f}")
print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")

MAE : 6,085,440.98
MSE : 237,084,532,471,795.53
RMSE: 15,397,549.56
R²  : 0.4078


## Evaluation Interpretation

The Linear Regression baseline achieves an R² of approximately 0.41 on the group-aware test set.

This result should not be compared directly with the earlier R² of 0.2359 because that earlier evaluation used a conventional random train/test split.

The current evaluation is more appropriate because identical feature groups cannot appear in both training and test sets.

The model is therefore treated as a baseline rather than a highly accurate final model.

In [12]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

results["Absolute_Error"] = (
    results["Actual"] - results["Predicted"]
).abs()

results.head(15)

,Actual,Predicted,Absolute_Error
0,20200000,1.913820e+07,1.061803e+06
1,10400000,1.066236e+07,2.623554e+05
2,18500000,1.263495e+07,5.865053e+06
3,42300000,3.774546e+07,4.554539e+06
4,27700000,3.137136e+07,3.671357e+06
5,5270000,1.088812e+07,5.618120e+06
6,28000000,3.137136e+07,3.371357e+06
7,11299999,3.152027e+07,2.022027e+07
8,15000000,2.707980e+07,1.207980e+07
9,7759999,1.066236e+07,2.902356e+06


## Conclusion

Linear Regression provides the baseline performance for this project.

The evaluation uses:

- the refined dataset
- `bhk`, `propertytype`, `location`, and `sqft`
- group-aware train/test splitting
- train-only model fitting
- a preprocessing pipeline
- MAE, MSE, RMSE, and R²

The feature group is:

`bhk + propertytype + location + sqft`

No feature group overlaps between the training and test sets.

Further model improvement should be evaluated against this corrected baseline.